<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #0f0c29 0%, #302b63 50%, #24243e 100%); border-radius: 15px; margin: 10px 0;'>
  <h1 style='color: #fff; margin: 0 0 8px 0; font-size: 2.2em;'>🎬 LTX-Video Generator</h1>
  <h3 style='color: #c0c0ff; margin: 0 0 5px 0; font-weight: 400;'>Lightning AI Studio — Q3_K_M (10.8 GB)</h3>
  <p style='color: #aaa; margin: 0;'>Text-to-Video • Image-to-Video | ComfyUI + LTX-2.3 GGUF</p>
</div>

---

### 🚀 Quick Start
1. Upload this notebook to **Lightning AI Studio**
2. Select **GPU machine** (T4) from top dropdown
3. **Cell 1** = setup + model download (~15 min first time)
4. **Cell 2** = Text-to-Video | **Cell 3** = Image-to-Video

In [ ]:
# @title ⚙️ Setup & Download Models (run once)
import os, subprocess
from pathlib import Path
from IPython.display import clear_output, display, HTML

HOME = os.path.expanduser('~')
COMFY = f'{HOME}/ComfyUI'

print('[1/3] Installing dependencies...')
!pip install -q torch torchvision torchaudio einops diffusers accelerate av spandrel albumentations onnx opencv-python onnxruntime tqdm ipywidgets

if not os.path.exists(COMFY):
    !git clone -q https://github.com/comfyanonymous/ComfyUI {COMFY}
!pip install -q -r {COMFY}/requirements.txt
!apt-get -y install -qq aria2 > /dev/null 2>&1

print('[2/3] Cloning custom nodes...')
%cd -q {COMFY}/custom_nodes
nodes = [
    "https://github.com/kijai/ComfyUI-KJNodes",
    "https://github.com/city96/ComfyUI-GGUF",
    "https://github.com/Lightricks/ComfyUI-LTXVideo/",
    "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite"
]
for node in nodes:
    name = node.split('/')[-1]
    if not os.path.exists(name):
        !git clone -q {node}
        req = f"{name}/requirements.txt"
        if os.path.exists(req):
            !pip install -q -r {req}

print('[3/3] Downloading Q3_K_M model (~10.8 GB)...')
def dl(url, dest, fname):
    Path(dest).mkdir(parents=True, exist_ok=True)
    fp = os.path.join(dest, fname)
    if not os.path.exists(fp):
        subprocess.run(['aria2c', '--console-log-level=error', '-c', '-x', '16', '-s', '16', '-k', '1M', '-d', dest, '-o', fname, url], check=True)

BASE = 'https://huggingface.co/unsloth/LTX-2.3-GGUF/resolve/main'
dl(f'{BASE}/distilled/ltx-2.3-22b-distilled-Q3_K_M.gguf', f'{COMFY}/models/unet', 'ltx-2.3-22b-distilled-Q3_K_M.gguf')
dl('https://huggingface.co/Dampfinchen/google-gemma-3-12b-it-qat-q4_0-gguf-small-fix/resolve/main/gemma-3-12b-it-q4_0_s.gguf', f'{COMFY}/models/text_encoders', 'gemma-3-12b-it-q4_0_s.gguf')
dl(f'{BASE}/text_encoders/ltx-2.3-22b-dev_embeddings_connectors.safetensors', f'{COMFY}/models/text_encoders', 'ltx-2.3-22b-dev_embeddings_connectors.safetensors')
dl(f'{BASE}/vae/ltx-2.3-22b-dev_video_vae.safetensors', f'{COMFY}/models/vae', 'ltx-2.3-22b-dev_video_vae.safetensors')
dl(f'{BASE}/vae/ltx-2.3-22b-dev_audio_vae.safetensors', f'{COMFY}/models/vae', 'ltx-2.3-22b-dev_audio_vae.safetensors')
dl('https://huggingface.co/Lightricks/LTX-2.3/resolve/main/ltx-2.3-spatial-upscaler-x2-1.0.safetensors', f'{COMFY}/models/latent_upscale_models', 'ltx-2.3-spatial-upscaler-x2-1.0.safetensors')
dl('https://huggingface.co/Lightricks/LTX-2.3/resolve/main/ltx-2.3-22b-distilled-lora-384.safetensors', f'{COMFY}/models/loras', 'ltx-2.3-22b-distilled-lora-384.safetensors')

clear_output()
print('✓ Setup complete! Run Cell 2 (T2V) or Cell 3 (I2V) to generate.')

In [ ]:
# @title 🎥 Text-to-Video
positive_prompt = "A cinematic shot of a futuristic sports car driving through a neon-lit cyberpunk city at night." # @param {type:"string"}
video_width = 832 # @param {type:"integer"}
video_height = 480 # @param {type:"integer"}
video_length_seconds = 4 # @param {type:"slider", min:1, max:10, step:1}

import json, urllib.request, time, os, glob, socket, subprocess
from PIL import Image
from tqdm.notebook import tqdm
from IPython.display import Video, display, HTML, clear_output

HOME = os.path.expanduser('~')
COMFY = f'{HOME}/ComfyUI'

safe_w = max(256, round(video_width / 32) * 32)
safe_h = max(256, round(video_height / 32) * 32)
if safe_w != video_width or safe_h != video_height:
    print(f'⚠️ Adjusted to {safe_w}x{safe_h} (must be divisible by 32)')

def is_server_running(port=8188):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('127.0.0.1', port)) == 0

if not is_server_running():
    print('Booting ComfyUI server...')
    os.chdir(COMFY)
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    subprocess.Popen(['python', 'main.py', '--dont-print-server', '--cache-none'])
    while not is_server_running():
        time.sleep(2)
    clear_output()
    print('✓ Server ready')

url = 'https://raw.githubusercontent.com/AICHUCKY/Comfyui-Workflows/AICHUCKY-patch-1/Ltx2.3%20.json'
req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
with urllib.request.urlopen(req) as r:
    wf = json.loads(r.read())

os.makedirs(f'{COMFY}/input/', exist_ok=True)
dummy = f'{COMFY}/input/IMG_20210620_151851479_processed.jpg'
Image.new('RGB', (safe_w, safe_h), color='black').save(dummy)

wf['292']['inputs']['value'] = safe_w
wf['293']['inputs']['value'] = safe_h
wf['285']['inputs']['value'] = 24
wf['290']['inputs']['value'] = True
wf['121']['inputs']['text'] = positive_prompt
wf['291']['inputs']['value'] = video_length_seconds
wf['137']['inputs']['sampler_name'] = 'lcm'
wf['360']['inputs']['sigmas'] = '1.0, 0.99375, 0.9875, 0.98125, 0.975, 0.909375, 0.725, 0.421875, 0.0'
wf['129']['inputs']['cfg'] = 1.0
wf['115']['inputs']['noise_seed'] = 43
wf['138']['inputs']['sampler_name'] = 'euler_cfg_pp'
wf['359']['inputs']['sigmas'] = '0.85, 0.7250, 0.4219, 0.0'
wf['103']['inputs']['cfg'] = 1.0
wf['114']['inputs']['noise_seed'] = 42

data = json.dumps({'prompt': wf}).encode('utf-8')
res = json.loads(urllib.request.urlopen(urllib.request.Request('http://127.0.0.1:8188/prompt', data=data)).read())
pid = res['prompt_id']

start = time.time()
with tqdm(desc='Rendering', bar_format='{desc} | {postfix}') as pbar:
    while True:
        elapsed = int(time.time() - start)
        m, s = divmod(elapsed, 60)
        pbar.set_postfix_str(f'{m:02d}:{s:02d}')
        try:
            hist = json.loads(urllib.request.urlopen(urllib.request.Request(f'http://127.0.0.1:8188/history/{pid}')).read())
            if str(pid) in hist:
                pbar.set_description('✅ Done')
                break
        except:
            pass
        time.sleep(2)

mp4s = glob.glob(f'{COMFY}/output/*.mp4') + glob.glob(f'{COMFY}/output/video/*.mp4')
if mp4s:
    display(Video(max(mp4s, key=os.path.getctime), embed=True, width=640))
else:
    print('⚠️ No video found')

In [ ]:
# @title 🖼️ Image-to-Video
# Upload an image to Lightning AI Studio first, then set the path below
input_image_path = "input.jpg" # @param {type:"string"}
positive_prompt = "The subject smiles and waves at the camera." # @param {type:"string"}
video_width = 832 # @param {type:"integer"}
video_height = 480 # @param {type:"integer"}
video_length_seconds = 3 # @param {type:"slider", min:1, max:10, step:1}

import json, urllib.request, time, os, glob, shutil, socket, subprocess
from tqdm.notebook import tqdm
from IPython.display import Video, display, HTML, clear_output

HOME = os.path.expanduser('~')
COMFY = f'{HOME}/ComfyUI'

safe_w = max(256, round(video_width / 32) * 32)
safe_h = max(256, round(video_height / 32) * 32)
if safe_w != video_width or safe_h != video_height:
    print(f'⚠️ Adjusted to {safe_w}x{safe_h}')

if not os.path.exists(input_image_path):
    print(f'❌ Image not found: {input_image_path}')
    print('Upload an image to the Studio first (drag & drop to file panel)')
else:
    img_name = os.path.basename(input_image_path)
    os.makedirs(f'{COMFY}/input/', exist_ok=True)
    shutil.copy(input_image_path, f'{COMFY}/input/{img_name}')

    def is_server_running(port=8188):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            return s.connect_ex(('127.0.0.1', port)) == 0

    if not is_server_running():
        print('Booting ComfyUI server...')
        os.chdir(COMFY)
        os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
        subprocess.Popen(['python', 'main.py', '--dont-print-server', '--cache-none'])
        while not is_server_running():
            time.sleep(2)
        clear_output()
        print('✓ Server ready')

    url = 'https://raw.githubusercontent.com/AICHUCKY/Comfyui-Workflows/AICHUCKY-patch-1/Ltx2.3%20.json'
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req) as r:
        wf = json.loads(r.read())

    wf['292']['inputs']['value'] = safe_w
    wf['293']['inputs']['value'] = safe_h
    wf['285']['inputs']['value'] = 24
    wf['290']['inputs']['value'] = False
    wf['167']['inputs']['image'] = img_name
    wf['121']['inputs']['text'] = positive_prompt
    wf['291']['inputs']['value'] = video_length_seconds
    wf['137']['inputs']['sampler_name'] = 'lcm'
    wf['360']['inputs']['sigmas'] = '1.0, 0.99375, 0.9875, 0.98125, 0.975, 0.909375, 0.725, 0.421875, 0.0'
    wf['129']['inputs']['cfg'] = 1.0
    wf['115']['inputs']['noise_seed'] = 43
    wf['138']['inputs']['sampler_name'] = 'euler_cfg_pp'
    wf['359']['inputs']['sigmas'] = '0.85, 0.7250, 0.4219, 0.0'
    wf['103']['inputs']['cfg'] = 1.0
    wf['114']['inputs']['noise_seed'] = 42

    data = json.dumps({'prompt': wf}).encode('utf-8')
    res = json.loads(urllib.request.urlopen(urllib.request.Request('http://127.0.0.1:8188/prompt', data=data)).read())
    pid = res['prompt_id']

    start = time.time()
    with tqdm(desc='Rendering', bar_format='{desc} | {postfix}') as pbar:
        while True:
            elapsed = int(time.time() - start)
            m, s = divmod(elapsed, 60)
            pbar.set_postfix_str(f'{m:02d}:{s:02d}')
            try:
                hist = json.loads(urllib.request.urlopen(urllib.request.Request(f'http://127.0.0.1:8188/history/{pid}')).read())
                if str(pid) in hist:
                    pbar.set_description('✅ Done')
                    break
            except:
                pass
            time.sleep(2)

    mp4s = glob.glob(f'{COMFY}/output/*.mp4') + glob.glob(f'{COMFY}/output/video/*.mp4')
    if mp4s:
        display(Video(max(mp4s, key=os.path.getctime), embed=True, width=640))
    else:
        print('⚠️ No video found')